# Unit 03｜采集函数

## Goal

在同一候选预测表上实现 Greedy、Uncertainty、UCB、PI、EI 和 Thompson Sampling。

本 Notebook 是确定性的人工教学实验，不是学习者已完成的研究，
也不是粘合剂实验结果。


## Setup

本单元不重新训练模型，避免把代理模型变化与采集函数变化混在一起。


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

pool_ids = np.array([f"M{i:02d}" for i in range(8)])
mean = np.array([6.9, 6.5, 6.2, 7.0, 6.7, 6.1, 6.8, 6.4])
std = np.array([0.2, 0.9, 1.3, 0.1, 0.5, 1.6, 0.4, 1.0])
best_observed = 6.6  # 只能来自已标注集
beta = 1.0
xi = 0.05
print("候选数:", len(pool_ids), "已观测最好值:", best_observed)


## Steps

按顺序执行。每个变量第一次出现时，先确认它的类型、形状和标签权限。


### 1. 计算确定性采集分数

Greedy 只看均值，Uncertainty 只看标准差，UCB 同时使用。


In [ ]:
greedy = mean
uncertainty = std
ucb = mean + beta * std


### 2. 计算 PI 与 EI

`safe_std` 防止除以零，`best_observed` 不得来自候选真实标签。


In [ ]:
improvement = mean - best_observed - xi
safe_std = np.maximum(std, 1e-12)
z = improvement / safe_std
pi = norm.cdf(z)
ei = improvement * norm.cdf(z) + std * norm.pdf(z)
ei = np.where(
    std > 0,
    ei,
    np.maximum(improvement, 0.0),
)


### 3. 固定种子进行边际近似 TS

这里只给出均值/标准差，无法恢复候选间协方差；因此是独立边际近似，不是 GP 联合后验的标准 TS。


In [ ]:
ts_rng = np.random.default_rng(2026)
thompson_marginal_approx = ts_rng.normal(mean, std)


### 4. 汇总策略选择

所有策略使用相同并列规则和相同候选 ID。


In [ ]:
scores = {
    "greedy": greedy,
    "uncertainty": uncertainty,
    "ucb": ucb,
    "pi": pi,
    "ei": ei,
    "thompson_marginal_approx": thompson_marginal_approx,
}

score_table = pd.DataFrame({"candidate_id": pool_ids, **scores})
selected = {}
for name, score in scores.items():
    order = np.lexsort((pool_ids.astype(str), -score))
    selected[name] = pool_ids[order[0]]

print(score_table.round(4).to_string(index=False))
print("各策略选择:", selected)


### 5. 查看 beta 如何改变 UCB

beta 在实验前声明；本表只用于理解敏感性，不根据隐藏标签选 beta。


In [ ]:
beta_table = pd.DataFrame({
    "candidate_id": pool_ids,
    "beta_0": mean,
    "beta_0.5": mean + 0.5 * std,
    "beta_2": mean + 2.0 * std,
})
print(beta_table.round(3).to_string(index=False))


## Checks

这些断言检查形状、预算和无重复等机械条件；通过断言不代表研究结论已经成立。


In [ ]:
assert set(selected) == set(scores)
assert all(candidate in pool_ids for candidate in selected.values())
assert np.all(ei >= 0)
assert np.all((pi >= 0) & (pi <= 1))
assert selected["greedy"] == "M03"
print("Unit 03 checks passed.")


## Next Steps

手算一个 UCB/EI 例子。Unit 04 会把一种采集函数放进多轮循环。
